In [21]:
from astroNN.datasets import load_galaxy10
from tensorflow.keras import utils
import numpy as np

# To load images and labels (will download automatically at the first time)
# First time downloading location will be ~/.astroNN/datasets/
images, labels = load_galaxy10()

# To convert the labels to categorical 10 classes
#labels = utils.to_categorical(labels, 10)

# To convert to desirable type
labels = labels.astype(np.float32)
images = images.astype(np.float32)

/Users/mariepilskog/.astroNN/datasets/Galaxy10_DECals.h5 was found!


In [22]:
from astroNN.datasets.galaxy10 import galaxy10cls_lookup
galaxy10cls_lookup(7)

'Unbarred Loose Spiral'

In [23]:
# Split into train and test
import numpy as np
from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(np.arange(labels.shape[0]), test_size=0.2)
train_images, train_labels, test_images, test_labels = images[train_idx], labels[train_idx], images[test_idx], labels[test_idx]

In [24]:
print(labels)

[0. 0. 0. ... 9. 9. 9.]


In [25]:
import numpy as np

def gini(y, classes):
    """Eq. (7.7): G = 1 - sum_k p_k^2."""
    p = np.array([np.mean(y == c) for c in classes])
    return 1.0 - np.sum(p**2)

def entropy(y, classes):
    """Eq. (7.8), with 0 log 0 = 0."""
    p = np.array([np.mean(y == c) for c in classes])
    p = p[p > 0]
    return -np.sum(p * np.log2(p))

def mse_impurity(y):
    """Eq. (7.12)."""
    return np.mean((y - y.mean())**2) if len(y) else 0.0

In [26]:
class Node:
    __slots__ = ("feature", "threshold",
                 "left", "right", "value", "n")

    def __init__(self, value=None, n=0):
        self.feature = self.threshold = None
        self.left = self.right = None
        self.value, self.n = value, n

In [27]:
class DecisionTree:
    """CART for regression and classification.
    task: "classification" or "regression";  criterion: "gini" or "entropy";
    max_features: if set, only m random features are tried at each split (more
                  only if none of them can split the node) -- this is what turns
                  bagging into a random forest (Part 6)."""

    def __init__(self, task="classification", criterion="gini", max_depth=None,
                 min_samples_split=2, min_samples_leaf=1, max_features=None, rng=None):
        self.task, self.criterion = task, criterion
        self.max_depth = np.inf if max_depth is None else max_depth
        self.min_samples_split, self.min_samples_leaf = min_samples_split, min_samples_leaf
        self.max_features = max_features
        self.rng = np.random.default_rng() if rng is None else rng

    def _impurity(self, y):
        if self.task == "regression":
            return mse_impurity(y)
        return gini(y, self.classes_) if self.criterion == "gini" else entropy(y, self.classes_)

    def _leaf_value(self, y):
        if self.task == "regression":
            return y.mean()                                   # the optimal constant
        counts = [np.sum(y == c) for c in self.classes_]
        return self.classes_[int(np.argmax(counts))]          # majority vote

In [28]:
def _best_split(self, X, y):
        n, p = X.shape
        features = np.arange(p)
        if self.max_features is not None and self.max_features < p:
            features = self.rng.permutation(p)                # random forest: a fresh random order
        parent = self._impurity(y)
        best_gain, best_j, best_t = -np.inf, None, None       # any admissible split beats "none"
        for rank, j in enumerate(features):
            if self.max_features is not None and rank >= self.max_features and best_j is not None:
                break                                         # m features tried, one works
            xs = X[:, j]
            uniq = np.unique(xs)                              # sorted distinct values
            if len(uniq) < 2:
                continue
            for t in (uniq[:-1] + uniq[1:]) / 2.0:            # candidate midpoints
                mask = xs <= t
                nl, nr = mask.sum(), n - mask.sum()
                if nl < self.min_samples_leaf or nr < self.min_samples_leaf:
                    continue
                child = (nl / n) * self._impurity(y[mask]) + (nr / n) * self._impurity(y[~mask])
                gain = parent - child                         # Eq. (7.10)
                if gain > best_gain + 1e-12:                  # first maximum wins a tie
                    best_gain, best_j, best_t = gain, j, t
        return best_gain, best_j, best_t

In [29]:
def _build(self, X, y, depth):
        node = Node(self._leaf_value(y), len(y))
        if (depth >= self.max_depth or len(y) < self.min_samples_split
                or len(np.unique(y)) == 1):           # stopping rules
            return node
        gain, j, t = self._best_split(X, y)
        if j is None:                             # no admissible split (gain 0 is fine)
            return node
        mask = X[:, j] <= t
        node.feature, node.threshold = j, t
        node.left = self._build(X[mask], y[mask], depth + 1)
        node.right = self._build(X[~mask], y[~mask], depth + 1)
        return node

def fit(self, X, y):
    X, y = np.asarray(X, dtype=float), np.asarray(y)
    if self.task == "classification":
        self.classes_ = np.unique(y)
    self.root_ = self._build(X, y, 0)
    return self

In [30]:
def predict(self, X):
        X = np.asarray(X, dtype=float)
        out = []
        for x in X:
            node = self.root_
            while node.feature is not None:   # walk to a leaf
                node = (node.left if x[node.feature] <= node.threshold
                        else node.right)
            out.append(node.value)
        return np.array(out)

In [31]:
DecisionTree._best_split = _best_split
DecisionTree._build = _build
DecisionTree.fit = fit
DecisionTree.predict = predict

In [32]:
def preprocess(imgs, size=16):
    """Downsample (N, 256, 256, 3) -> (N, size, size, 3) by block-averaging, then flatten."""
    N, H, W, C = imgs.shape
    f = H // size                                   # e.g. 256 // 16 = 16
    small = imgs.reshape(N, size, f, size, f, C).mean(axis=(2, 4))   # (N, size, size, 3)
    return small.reshape(N, -1) / 255.0             # (N, size*size*3)

X_train = preprocess(train_images, size=16)         # 16*16*3 = 768 features
X_test  = preprocess(test_images,  size=16)
print(X_train.shape, X_test.shape)

(14188, 768) (3548, 768)


In [38]:
rng = np.random.default_rng(0)
sub = rng.choice(len(X_train), 10000, replace=False)
X_tr, y_tr = X_train[sub], train_labels[sub]

import time
tree = DecisionTree(task="classification", criterion="gini",
                    max_depth=7, min_samples_leaf=7)
t0 = time.time()
tree.fit(X_tr, y_tr)
print(f"fit time: {time.time()-t0:.1f}s")
print("train acc:", np.mean(tree.predict(X_tr) == y_tr))
print("test acc: ", np.mean(tree.predict(X_test) == test_labels))


fit time: 4228.3s
train acc: 0.4548
test acc:  0.37767756482525366
